# The Emulated Coprocessor

A quantum-classical LabOne Q experiment running on a heterogeneous control system streams data to and from a coprocessor during runtime. An emulated coprocessor runs as a Python function, in the same process that runs the controller. It is fed with the same emulated measurement data you also can find in `acquired_results`. It exists to teach and demonstrate the LabOne Q programming model for a heterogeneous control system.

The emulated coprocessor does not model a real coprocessor:

- Responses are instantaneous. There is no timing model.
- Return values cannot steer the sequence. `match` on a coprocessor variable
  always takes the `case(0)` branch, and `do_until` runs its body exactly once.
- Waveform updates can be declared, but the coprocessor cannot actually exercise them.


## An emulated setup

The emulated coprocessor must be added to the device setup by hand by calling `add_emulated_coprocessor`.
It returns the inventory entry that the experiment's coprocessor must be mapped to.

In [ ]:
from laboneq.dsl.coprocessor.builtins import mark_stale
from laboneq.dsl.experiment.builtins import *
from laboneq.simple import *
from zhinst_zqcs.emulation import EmulatedCoprocessorKernel, add_emulated_coprocessor

descriptor = """\
instruments:
  ZQCS:
  - address: http://DEV_ZQCS
    uid: zqcs
connections:
  zqcs:
    - iq_signal: q0/drive_line
      port: "1:3:1:1"
    - acquire_signal: q0/acquire_line
      port: "1:3:1:2"
    - iq_signal: q0/measure_line
      port: "1:3:1:3"
"""

setup = DeviceSetup.from_descriptor(
    yaml_text=descriptor, server_host="localhost", server_port="8004"
)
emulated_coprocessor = add_emulated_coprocessor(setup)

session = Session(setup)
session.connect(do_emulation=True)

## Declaring the exchange

The `@experiment(coprocessors=["analyzer"])` declaration names the coprocessor.

`register_stream` declares one typed channel in one direction: `dst=` is outbound
(control system to coprocessor), `src=` is inbound. Always pass a `uid`, because that
is the key with which your emulated kernel looks up the stream.

An outbound field bound with `set_handles` carries acquired data. At every `send`, the
driver reads each bound handle's result at the current real-time index and puts them
into the packet. An inbound field bound with `as_variable` receives data from the
kernel, and `.log(handle=...)` on that variable puts it into
`results.variable_results`.

### send, mark_stale

`send(stream)` transmits one packet to the kernel. `mark_stale(variable)` and `expect(is_live(variable))` signal that the coprocessor is expected to provide new data. The coprocessor must emit exactly one datum for each `mark_stale` for the data to be written to the log. The pairing is strict.

In [ ]:
readout_kernel = pulse_library.const(length=100e-9)
N_SHOTS = 4


@experiment(signals=["measure", "acquire"], coprocessors=["analyzer"])
def roundtrip():
    measurements = register_stream(
        dst="analyzer", schema=Struct(shot=IqDataPacked), uid="measurements"
    )
    measurements.fields["shot"].set_handles(["q0"])

    verdicts = register_stream(
        src="analyzer", schema=Struct(verdict=Int32), uid="verdicts"
    )
    verdict = verdicts.fields["verdict"].as_variable(name="verdict", initial=0)
    verdict.log(handle="verdicts")

    with acquire_loop_rt(
        count=N_SHOTS,
        uid="shots",
        averaging_mode=AveragingMode.SINGLE_SHOT,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        with section(uid="readout"):
            acquire("acquire", handle="q0", kernel=readout_kernel)
            delay(signal="acquire", time=2e-9)
        with section(uid="exchange"):
            send(measurements)
            mark_stale(verdict)
            # [Omitted here: waiting time. Emulated coprocessor is instantaneous.]
            expect(is_live(verdict))

    map_signal(
        "measure", setup.logical_signal_groups["q0"].logical_signals["measure_line"]
    )
    map_signal(
        "acquire", setup.logical_signal_groups["q0"].logical_signals["acquire_line"]
    )

## The kernel

A kernel subclasses `EmulatedCoprocessorKernel` and implements `async def run(self, ctx)`.
The streams can be accessed through `ctx.streams[uid]`.

`await ctx.streams[uid].recv()` suspends until the next packet on that stream arrives.
`ctx.streams[uid].emit(field=value)` sends a value back.

A packet's fields are attributes: `packet.shot`. A field of a *packed* type is a
dict: it holds one value per handle bound with
`set_handles`.

In [ ]:
class Discriminator(EmulatedCoprocessorKernel):
    """Classify each shot by its acquisition index."""

    async def run(self, ctx):
        while True:
            packet = await ctx.streams["measurements"].recv()
            ctx.streams["verdicts"].emit(verdict=int(packet.shot["q0"].imag >= 2))

## Attaching the kernel

In general, a real coprocessor's payload is opaque bytes, which LabOne Q forwards untouched to the
real coprocessor. The _emulated_ co-processor's payload however has semantic meaning to LabOne Q:
it is a UTF-8 encoded string naming the class that implements the kernel. The syntax is
`<module>:<attribute>`. Use `__main__:<name of the class>` if the kernel is defined inline in the current notebook or script.

In [ ]:
exp = roundtrip()
exp.map_coprocessor("analyzer", emulated_coprocessor)

compiled = session.compile(exp)
compiled.set_coprocessor_payload("analyzer", b"__main__:Discriminator")

results = session.run(compiled)

## Logging data from the kernel

Emitted values land in `results.variable_results`, keyed by the log handle passed to
`.log()`. Each entry carries `data`, an n-dimensional numpy array.
It is shaped after the nested loop structure, like the acquired results.

In [ ]:
acquired = results.acquired_results["q0"]
logged = results.variable_results["verdicts"]

print("acquired :", acquired.data)
print("logged   :", logged.data)
print("axes     :", logged.axis_name, "vs acquired", acquired.axis_name)

# The axes of a logged variable equal the axes of a handle acquired in the same
# loops, so the two can be plotted against one another.
assert logged.axis_name == acquired.axis_name

## Shape of the log

A logged variable's shape is defined by loops enclosing its `mark_stale` sites.
To build some intuition, it might help to replace `mark_stale` by `acquire`, and think of the
resulting shape of the results for that acquisition handle. Logged variables work the same.

In the example below, `mark_stale` is located in a sweep nested in the averaging loop,
so the log becomes two-dimensional.

In [ ]:
drive_pulse = pulse_library.gaussian(uid="drive", length=100e-9, amplitude=1.0)


@experiment(signals=["drive", "measure", "acquire"], coprocessors=["analyzer"])
def swept():
    measurements = register_stream(
        dst="analyzer", schema=Struct(shot=IqDataPacked), uid="measurements"
    )
    measurements.fields["shot"].set_handles(["q0"])

    verdicts = register_stream(
        src="analyzer", schema=Struct(verdict=Int32), uid="verdicts"
    )
    verdict = verdicts.fields["verdict"].as_variable(name="verdict", initial=0)
    verdict.log(handle="verdicts")

    with acquire_loop_rt(
        count=2,
        uid="shots",
        averaging_mode=AveragingMode.SINGLE_SHOT,
        acquisition_type=AcquisitionType.INTEGRATION,
    ):
        with sweep_range(0.1, 1.0, 3, uid="amplitude", axis_name="amplitude") as amp:
            with section(uid="drive_sec"):
                play("drive", drive_pulse, amplitude=amp)
            with section(uid="readout"):
                acquire("acquire", handle="q0", kernel=readout_kernel)
                delay(signal="acquire", time=2e-9)
            with section(uid="exchange"):
                send(measurements)
                mark_stale(verdict)
                # [Omitted here: waiting time. Emulated coprocessor is instantaneous.]
                expect(is_live(verdict))

    map_signal("drive", setup.logical_signal_groups["q0"].logical_signals["drive_line"])
    map_signal(
        "measure", setup.logical_signal_groups["q0"].logical_signals["measure_line"]
    )
    map_signal(
        "acquire", setup.logical_signal_groups["q0"].logical_signals["acquire_line"]
    )


class IndexEcho(EmulatedCoprocessorKernel):
    async def run(self, ctx):
        while True:
            packet = await ctx.streams["measurements"].recv()
            ctx.streams["verdicts"].emit(verdict=int(packet.shot["q0"].imag))


swept_exp = swept()
swept_exp.map_coprocessor("analyzer", emulated_coprocessor)
swept_compiled = session.compile(swept_exp)
swept_compiled.set_coprocessor_payload("analyzer", b"__main__:IndexEcho")
swept_results = session.run(swept_compiled)

swept_logged = swept_results.variable_results["verdicts"]
print("shape:", swept_logged.data.shape, "axes:", swept_logged.axis_name)
print(swept_logged.data)

## Failures

Each kernel below violates the contract differently. The controller wraps any failure
during result collection in a generic `RuntimeError("operation failed")`, so the
message explaining what went wrong sits on `__cause__` rather than on the exception
you catch.

In [ ]:
class Silent(EmulatedCoprocessorKernel):
    """Never emits, so expect finds the variable still stale."""

    async def run(self, ctx):
        while True:
            await ctx.streams["measurements"].recv()


class Chatty(EmulatedCoprocessorKernel):
    """Emits two values for one mark_stale."""

    async def run(self, ctx):
        while True:
            await ctx.streams["measurements"].recv()
            ctx.streams["verdicts"].emit(verdict=1)
            ctx.streams["verdicts"].emit(verdict=2)


class Misspelled(EmulatedCoprocessorKernel):
    async def run(self, ctx):
        while True:
            await ctx.streams["measurements"].recv()
            ctx.streams["verdicts"].emit(verdikt=1)


class WrongDirection(EmulatedCoprocessorKernel):
    """Calls recv() on the inbound stream."""

    async def run(self, ctx):
        while True:
            await ctx.streams["verdicts"].recv()


def root_cause(exc):
    while exc.__cause__ is not None or exc.__context__ is not None:
        exc = exc.__cause__ or exc.__context__
    return exc


for kernel_class in (Silent, Chatty, Misspelled, WrongDirection):
    compiled.set_coprocessor_payload(
        "analyzer", f"__main__:{kernel_class.__name__}".encode()
    )
    try:
        session.run(compiled)
    except Exception as exc:
        print(f"{kernel_class.__name__:16} {root_cause(exc)}")

## Limitations

The compiler rejects an emulated coprocessor experiment that

- uses any averaging mode other than `AveragingMode.SINGLE_SHOT`,
- sweeps in near time, or chunks a real-time sweep,
- calls `mark_stale` on one variable at two different loop depths.
